# 02 · EDA (images) — what the scans actually look like

Run `python src/extract_headers.py --stageII <dir> --ctc <dir>` first. It writes `data/processed/dicom_series.parquet` (one row per downloaded series).

| Section | Question | Paper section |
|---|---|---|
| 1 | Did every selected series load? | §4.2.6 |
| 2 | Slice thickness, spacing and matrix: confirm 5 mm vs about 1 mm | §4.2.2 |
| 3 | Dose, kernel and contrast: other differences between the sources | §4.2.2 |
| 4 | Fix the unknown prone/supine positions | §4.2.5 |
| 5 | Intensity (HU) distributions by source | §4.2.2, §4.4 |
| 6 | Sample slices | §4.2 (figure) |
| 7 | Can intensity alone tell the sources apart? (source-prediction check) | §4.4 |

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FIG, TAB = ROOT / "reports" / "figures", ROOT / "reports" / "tables"; FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
d = pd.read_parquet(ROOT / "data/processed/dicom_series.parquet")
ser = pd.read_parquet(ROOT / "data/processed/series.parquet")
INK, MUTED, GRID, SURF, BLUE, ORANGE, GRAY = "#0b0b0b", "#52514e", "#e6e5e1", "#ffffff", "#2a78d6", "#eb6834", "#9b9a95"
COL = {"StageII": ORANGE, "CTC": BLUE}
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "text.color": INK,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": GRID, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False})
def tidy(ax, grid="y"):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    if grid: ax.grid(axis=grid, color=GRID, lw=.8); ax.set_axisbelow(True)
def save(fig, n): fig.savefig(FIG / f"{n}.png", dpi=150, bbox_inches="tight"); plt.show()
f = d[d.found == True].copy()
print(f"{len(f)} of {len(d)} series loaded")

## 1 · Completeness &nbsp; → §4.2.6

In [ ]:
t = d.groupby("source").agg(series=("series_uid", "size"), loaded=("found", "sum"), patients=("patient_id", "nunique"))
t["images"] = f.groupby("source").n_files.sum(); t.to_csv(TAB / "img_01_completeness.csv"); display(t)
miss = d[d.found != True]; print("not loaded:", len(miss))

## 1b · Which series are real colon volumes? &nbsp; → §4.2.5
Forty CT Colonography patients (IDs starting `CTC-`) have no series descriptions, so names cannot tell volumes from other series. Each has two large volumes plus two small series of 50–70 slices with repeated slice positions (z jumps of 200–400 mm). Rule used from here on: **a CTC series is a volume if it has at least 200 slices and a median slice spacing of 2.5 mm or less**. StageII series are all kept.

In [ ]:
MIN_SLICES, MAX_STEP = 200, 2.5
f["kind"] = np.where(f.source == "StageII", "StageII volume",
             np.where((f.n_files >= MIN_SLICES) & (f.z_step_median <= MAX_STEP), "CTC volume", "CTC other (excluded)"))
f["id_style"] = np.where(f.patient_id.str.startswith("CTC-"), "CTC- IDs", np.where(f.source == "StageII", "StageII", "1.3.6… IDs"))
t = pd.crosstab([f.source, f.id_style], f.kind); t.to_csv(TAB / "img_01b_series_kinds.csv"); display(t)
ex = f[f.kind == "CTC other (excluded)"]
print("excluded series:", len(ex), "| slices:", ex.n_files.describe()[["min", "50%", "max"]].tolist(), "| spacing median:", ex.z_step_median.median())
display(ex[["patient_id", "desc", "n_files", "z_step_median", "z_step_max", "n_duplicate_z"]].head(12))
vol = f[f.kind != "CTC other (excluded)"].copy()
pv = vol[vol.source == "CTC"].groupby("patient_id").size()
allp = f[f.source == "CTC"].patient_id.unique()
print(f"CTC patients with a volume: {len(pv)} of {len(allp)}; with fewer than 2 volumes: {(pv < 2).sum()}; with more than 2: {(pv > 2).sum()}")
big_gap = vol[(vol.source == "CTC") & (vol.z_step_max > 5)]
print("volumes with a coverage gap > 5 mm:", len(big_gap)); display(big_gap[["patient_id", "desc", "n_files", "z_step_max"]])
f = vol   # everything below uses volumes only

## 2 · Geometry: thickness, spacing, matrix &nbsp; → §4.2.2

In [ ]:
f["z_step"] = f.z_step_median.round(2)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for s, c in COL.items():
    x = f[f.source == s]
    axs[0].hist(np.clip(x.z_step_median.dropna(), 0, 8), bins=np.arange(0, 8.25, .25), color=c, alpha=.6, label=s, edgecolor=SURF)
    axs[1].hist(x.pixel_mm.dropna(), bins=20, color=c, alpha=.6, label=s, edgecolor=SURF)
    axs[2].hist(x.n_files, bins=20, color=c, alpha=.6, label=s, edgecolor=SURF)
for a, tt, xl in zip(axs, ["Slice spacing", "In-plane pixel size", "Slices per volume"], ["mm between slices (median)", "mm", "slices"]):
    a.set_title(tt); a.set_xlabel(xl); tidy(a)
axs[0].legend(); axs[0].set_ylabel("Series"); fig.tight_layout(); save(fig, "img_02_geometry")
g = f.groupby("source").agg(series=("series_uid", "size"), z_step_median=("z_step_median", "median"), z_step_max=("z_step_max", "max"),
    thickness_median=("slice_thickness", "median"), pixel_mm=("pixel_mm", "median"), matrix=("rows", lambda x: x.mode().iloc[0]),
    slices_median=("n_files", "median"), z_extent_mm=("z_extent_mm", "median"), dup_z=("n_duplicate_z", "sum")).round(2)
g.to_csv(TAB / "img_02_geometry.csv"); display(g)

**Reading it.** Fill in once run: the expected gap is about 5 mm for StageII against about 1 mm (or less) for CTC. A large `z_step_max` flags series with missing slices, and `dup_z` counts duplicate positions.

## 3 · Dose, kernel, contrast &nbsp; → §4.2.2

In [ ]:
cols = ["kvp", "tube_current", "kernel", "contrast", "manufacturer"]
rows = []
for s in COL:
    x = f[f.source == s]
    rows.append({"source": s, "kVp (mode)": x.kvp.mode().iloc[0] if x.kvp.notna().any() else None,
                 "tube current, median mA": x.tube_current.median(),
                 "kernels (top 3)": "; ".join(f"{k} ×{n}" for k, n in x.kernel.value_counts().head(3).items()),
                 "contrast agent recorded": f"{x.contrast.notna().sum()} of {len(x)}"})
t = pd.DataFrame(rows).set_index("source"); t.to_csv(TAB / "img_03_acquisition.csv"); display(t)

## 4 · Resolving unknown prone/supine position &nbsp; → §4.2.5

In [ ]:
m = f.merge(ser[["series_uid", "position"]], on="series_uid", how="left")
m["pp"] = m.patient_position.astype(str).str.upper().str.strip()
tab = pd.crosstab(m.position.fillna("n/a"), m.pp); tab.to_csv(TAB / "img_04_position_vs_header.csv"); display(tab)
print("HFP / FFP = prone, HFS / FFS = supine. Series with unknown position but a usable header:",
      int(((m.position == "unknown") & m.pp.isin(["HFP", "FFP", "HFS", "FFS"])).sum()))

## 5 · Intensity (HU) by source &nbsp; → §4.2.2, §4.4

In [ ]:
if "hu_p50" in f:
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
    for a, c_, tt in zip(axs, ["hu_p1", "hu_p50", "air_frac"], ["1st percentile HU (air / lumen)", "Median HU (soft tissue)", "Share of voxels below -900 HU"]):
        for s, c in COL.items():
            a.hist(f.loc[f.source == s, c_].dropna(), bins=25, color=c, alpha=.6, label=s, edgecolor=SURF)
        a.set_title(tt); tidy(a)
    axs[0].legend(); axs[0].set_ylabel("Series"); fig.tight_layout(); save(fig, "img_05_hu_by_source")
    f.groupby("source")[["hu_p1", "hu_p50", "hu_p99", "air_frac"]].describe().T.round(1).to_csv(TAB / "img_05_hu_summary.csv")
else: print("no pixel statistics (see pixel_error column)")

## 6 · Sample slices &nbsp; → §4.2 figure

In [ ]:
import pydicom
def load(path):
    ds = pydicom.dcmread(path); return ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
def win(a, lo=-160, hi=240): return np.clip((a - lo) / (hi - lo), 0, 1)
if "mid_file" in f:
    pick = pd.concat([f[(f.source == "StageII")].dropna(subset=["mid_file"]).head(3), f[(f.source == "CTC")].dropna(subset=["mid_file"]).head(3)])
    fig, axs = plt.subplots(2, 3, figsize=(10, 6.6))
    for a, (_, r) in zip(axs.ravel(), pick.iterrows()):
        a.imshow(win(load(r.mid_file)), cmap="gray"); a.set_title(f"{r.source}  ·  {r.z_step_median:.1f} mm", fontsize=9); a.axis("off")
    fig.tight_layout(); save(fig, "img_06_sample_slices")

## 7 · Source-prediction check: raw vs harmonised &nbsp; → §4.4
Run `python src/preprocess.py --out <dir>` first. A logistic regression tries to tell StageII from CT Colonography from **intensity statistics only** (no geometry). It is trained on the frozen `train` split and scored on `test`, once on raw values and once on harmonised volumes (orientation fixed, 5 mm slabs, 1.5 mm pixels, padding and table masked, HU clipped to [-1024, 1000], cropped to the body). The drop in AUC is the evidence that harmonisation removed scanner-specific cues; an AUC of 0.5 would mean the sources can no longer be separated by intensity.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
pp_path = ROOT / "data/processed/preproc_series.parquet"
if not pp_path.exists():
    print("preproc_series.parquet not found: run  python src/preprocess.py --out <folder>  first")
else:
    pp = pd.read_parquet(pp_path); pp = pp[pp.ok == True]
    g = f.merge(pp, on="series_uid")
    SETS = {"Raw: 4 summaries": ["hu_p1", "hu_p50", "hu_p99", "air_frac"],
            "Raw: 32-bin histogram": [c for c in g if c.startswith("raw_h")],
            "Harmonised: 4 summaries": ["pre_p1", "pre_p50", "pre_p99", "pre_air_frac"],
            "Harmonised: 32-bin histogram": [c for c in g if c.startswith("pre_h")]}
    tr, te = g[g.split == "train"], g[g.split == "test"]
    ytr, yte = (tr.source == "CTC").to_numpy(), (te.source == "CTC").to_numpy()
    print(f"train {len(tr)} series ({ytr.sum()} CTC), test {len(te)} series ({yte.sum()} CTC)")
    rng = np.random.default_rng(0)
    def boot(y, p, B=2000):
        out = []
        for _ in range(B):
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i): out.append(roc_auc_score(y[i], p[i]))
        return np.percentile(out, [2.5, 97.5])
    rows = []
    for name, cols in SETS.items():
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=1.0)).fit(tr[cols], ytr)
        p = clf.predict_proba(te[cols])[:, 1]; lo, hi = boot(yte, p)
        cv = cross_val_predict(clf, tr[cols], ytr, groups=tr.patient_id, cv=GroupKFold(5), method="predict_proba")[:, 1]
        rows.append({"features": name, "test_AUC": roc_auc_score(yte, p), "CI_low": lo, "CI_high": hi,
                     "test_balanced_accuracy": balanced_accuracy_score(yte, p > .5), "train_CV_AUC": roc_auc_score(ytr, cv),
                     "n_train": len(tr), "n_test": len(te)})
    res = pd.DataFrame(rows).round(3); res.to_csv(TAB / "img_07_source_prediction.csv", index=False); display(res)
    fig, ax = plt.subplots(figsize=(8.6, 3.6)); y = np.arange(len(res))[::-1]
    ax.barh(y, res.test_AUC, height=.55, color=[ORANGE if n.startswith("Raw") else BLUE for n in res.features])
    ax.errorbar(res.test_AUC, y, xerr=[res.test_AUC - res.CI_low, res.CI_high - res.test_AUC], fmt="none", ecolor=INK, capsize=3, lw=1)
    for yi, v in zip(y, res.test_AUC): ax.text(1.03, yi, f"{v:.2f}", va="center", fontweight="bold")
    ax.axvline(.5, color=MUTED, lw=1, ls=(0, (4, 3))); ax.text(.51, -.62, "0.5 = cannot tell sources apart", color=MUTED, ha="left", fontsize=9)
    ax.set_yticks(y); ax.set_yticklabels(res.features); ax.set_xlim(0, 1.12); ax.set_ylim(-.8, len(res) - .4); ax.set_xlabel("AUC for telling StageII from CT Colonography (test split, 95% CI)")
    ax.set_title("How much harmonisation reduces the source signal"); tidy(ax, "x"); save(fig, "img_07_source_prediction")